# 02 — Modelagem Preditiva
**Tech Challenge Fase 3 — FIAP IA Scientist**

Este notebook documenta o processo de modelagem para prever municípios
em risco de não atingir a meta de alfabetização.

## Design temporal t → t+1

- **Features**: dados observados em 2023
- **Target**:  (taxa_2024 < 60%)
- **Motivação**: elimina data leakage estrutural da v1 (ROC-AUC 0.997 → 0.910)

## Tratamento de data leakage

Variáveis **removidas** por causar vazamento:
-  de 2024 → é o target
-  de 2024 → compõem a fórmula do target
-  de 2024 → derivado dos níveis

Variáveis de 2023 **podem** entrar como features — são o histórico.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import json
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    recall_score, classification_report, confusion_matrix
)
import seaborn as sns

plt.style.use("seaborn-v0_8")

## 1. Carregamento do dataset temporal

In [ ]:
caminho = Path("data/processed/dataset_enriquecido_v2.parquet")
if not caminho.exists():
    caminho = Path("data/processed/dataset_modelagem_v2.parquet")

df = pd.read_parquet(caminho)
print(f"Shape: {df.shape}")
print(f"
Distribuição do target:")
print(df["em_risco_2024"].value_counts())
print(f"
Balanceamento: {df["em_risco_2024"].mean()*100:.1f}% em risco")

## 2. Carregamento do modelo treinado

O modelo foi treinado em  e salvo com .
Carregamos aqui para análise — **sem retreinar**.

In [ ]:
pipeline = joblib.load("models/modelo_final.joblib")

with open("models/metadata.json", encoding="utf-8") as f:
    metadata = json.load(f)

print(f"Modelo: {metadata['modelo']}")
print(f"Threshold: {metadata['threshold']:.3f}")
print(f"
Métricas no teste:")
for k, v in metadata["metricas_teste"].items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

## 3. Avaliação no conjunto de teste

In [ ]:
features_num = metadata["features_numericas"]
features_cat = metadata["features_categoricas"]
features_all = [f for f in features_num + features_cat if f in df.columns]

X = df[features_all]
y = df["em_risco_2024"]

_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

threshold = metadata["threshold"]
y_prob = pipeline.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= threshold).astype(int)

print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, y_prob):.4f}")
print(f"Recall:  {recall_score(y_test, y_pred):.4f}")
print(f"
{classification_report(y_test, y_pred, target_names=['Nao risco', 'Em risco'])}")

## 4. Visualizações

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)
axes[0].plot(fpr, tpr, color="steelblue", linewidth=2, label=f"AUC = {auc:.4f}")
axes[0].plot([0, 1], [0, 1], "k--")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("Curva ROC")
axes[0].legend()

# Matriz de confusão
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[1],
            xticklabels=["Nao risco", "Em risco"],
            yticklabels=["Nao risco", "Em risco"])
axes[1].set_title(f"Matriz de Confusão
(threshold={threshold:.3f})")
axes[1].set_ylabel("Real")
axes[1].set_xlabel("Previsto")

plt.tight_layout()
plt.show()

## 5. Análise do threshold

O threshold foi ajustado para garantir **recall >= 0.85**.
Para política pública, o custo de não identificar um município em risco
é maior que o custo de um falso alarme.

In [ ]:
prec_curve, rec_curve, thresholds = precision_recall_curve(y_test, y_prob)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, prec_curve[:-1], label="Precision", color="steelblue")
ax.plot(thresholds, rec_curve[:-1], label="Recall", color="orange")
ax.axvline(x=threshold, color="red", linestyle="--",
           label=f"Threshold escolhido ({threshold:.3f})")
ax.axhline(y=0.85, color="green", linestyle="--", alpha=0.5,
           label="Recall alvo (0.85)")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("Precision e Recall por Threshold")
ax.legend()
plt.tight_layout()
plt.show()

## Conclusões

- **ROC-AUC 0.9103**: modelo com capacidade discriminativa real
- **Recall 0.86**: captura 86% dos municípios em risco
- **Design temporal correto**: features 2023 → target 2024
- **Threshold ajustado**: 0.458 maximiza recall mantendo precisão aceitável
- Resultado defensável e explicável para gestores públicos